In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("forestfires.csv")

print(df.head())
print(df.columns)

   X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0
Index(['X', 'Y', 'month', 'day', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH',
       'wind', 'rain', 'area'],
      dtype='object')


In [2]:
# Create category column
def classify(area):
    if area == 0:
        return "NotAffected"
    elif area <= 50:
        return "PartiallyAffected"
    else:
        return "MostlyAffected"

df['DamageLevel'] = df['area'].apply(classify)

# Create subsets
not_affected = df[df['DamageLevel'] == "NotAffected"]
partial = df[df['DamageLevel'] == "PartiallyAffected"]
mostly = df[df['DamageLevel'] == "MostlyAffected"]

print(not_affected.head())
print(partial.head())
print(mostly.head())

   X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area  \
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0   
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0   
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0   
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0   
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0   

   DamageLevel  
0  NotAffected  
1  NotAffected  
2  NotAffected  
3  NotAffected  
4  NotAffected  
     X  Y month  day  FFMC    DMC     DC   ISI  temp  RH  wind  rain  area  \
138  9  9   jul  tue  85.8   48.3  313.4   3.9  18.0  42   2.7   0.0  0.36   
139  1  4   sep  tue  91.0  129.5  692.6   7.0  21.7  38   2.2   0.0  0.43   
140  2  5   sep  mon  90.9  126.5  686.5   7.0  21.9  39   1.8   0.0  0.47   
141  1  2   aug  wed  95.5   99.9  513.3  13.2  23.3  31   4.5   0.0  0.55   
142  8  6   aug  fri  90.1  108.0  529.8  12.5  21.2  51   8.9 

In [3]:
# Merge NotAffected and PartiallyAffected
merged_data = pd.concat([not_affected, partial])

print(merged_data.shape)

(493, 14)


In [4]:
# Sort by temperature, wind, and area
sorted_data = df.sort_values(by=['temp', 'wind', 'area'])

print(sorted_data[['temp','wind','area']].head())

     temp  wind  area
280   2.2   4.9  9.27
282   4.2   4.0  0.00
465   4.6   0.9  6.84
463   4.6   6.3  5.39
279   4.6   8.5  9.77


In [5]:
# Transpose dataset
transpose_data = df.transpose()

print(transpose_data.head())

        0     1     2     3     4     5     6     7     8     9    ...   507  \
X         7     7     7     8     8     8     8     8     8     7  ...     2   
Y         5     4     4     6     6     6     6     6     6     5  ...     4   
month   mar   oct   oct   mar   mar   aug   aug   aug   sep   sep  ...   aug   
day     fri   tue   sat   fri   sun   sun   mon   mon   tue   sat  ...   fri   
FFMC   86.2  90.6  90.6  91.7  89.3  92.3  92.3  91.5  91.0  92.5  ...  91.0   

        508   509   510   511   512   513   514   515   516  
X         1     5     6     8     4     2     7     1     6  
Y         2     4     5     6     3     4     4     4     3  
month   aug   aug   aug   aug   aug   aug   aug   aug   nov  
day     fri   fri   fri   sun   sun   sun   sun   sat   tue  
FFMC   91.0  91.0  91.0  81.6  81.6  81.6  81.6  94.4  79.5  

[5 rows x 517 columns]


In [7]:
# Convert to long format
melted_data = pd.melt(df,
                      id_vars=['month'],
                      value_vars=['temp', 'wind', 'area'],
                      var_name='Attribute',
                      value_name='Value')

print(melted_data.head())

  month Attribute  Value
0   mar      temp    8.2
1   oct      temp   18.0
2   oct      temp   14.6
3   mar      temp    8.3
4   mar      temp   11.4


In [8]:
# Convert back to wide format
wide_data = melted_data.pivot_table(index='month',
                                    columns='Attribute',
                                    values='Value',
                                    aggfunc='mean')

print(wide_data)

Attribute       area       temp      wind
month                                    
apr         8.891111  12.044444  4.666667
aug        12.489076  21.631522  4.086413
dec        13.330000   4.522222  7.644444
feb         6.275000   9.635000  3.755000
jan         0.000000   5.250000  2.000000
jul        14.369687  22.109375  3.734375
jun         5.841176  20.494118  4.135294
mar         4.356667  13.083333  4.968519
may        19.240000  14.650000  4.450000
nov         0.000000  11.800000  4.500000
oct         6.638000  17.093333  3.460000
sep        17.942616  19.612209  3.557558
